## Cell 1 — Install Dependencies

# Security LLM Finetuning Notebook
**Models**: SmolLM2-360M-Instruct | Qwen3-0.6B | Gemma-3-270m-it  
**Hardware**: Kaggle T4 GPU (16 GB VRAM)  
**Method**: QLoRA (4-bit NF4) + LoRA (r=16, alpha=32)  
**Benchmarks**: A) Merged finetuning → per-dataset eval | B) Individual dataset finetuning  
**Metrics**: BLEU, ROUGE-1/2/L, Token-F1, Macro-F1, Accuracy

In [ ]:
# !pip install -q transformers
# !pip install -q datasets
!pip install -q peft
!pip install -q trl
!pip install -q bitsandbytes>=0.46.1
# !pip install -q accelerate
# !pip install -q sentencepiece
# !pip install -q protobuf
!pip install -q evaluate
!pip install -q rouge_score
!pip install -q nltk
!pip install -q sacrebleu
# !pip install -q scikit-learn
# !pip install -q pandas
# !pip install -q numpy
# !pip install -q tensorboard
# !pip install -q tqdm
# !pip install -q torch

## Cell 2 — All Imports

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc
import re
import json
import math
import copy
import random
import logging
import warnings
import itertools
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any

import nltk
import numpy as np
import pandas as pd
import torch
import evaluate
from tqdm.auto import tqdm

from datasets import (
    load_dataset,
    Dataset,
    DatasetDict,
    concatenate_datasets,
    interleave_datasets,
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    TrainerCallback,
    TrainerState,
    TrainerControl,
    DataCollatorForSeq2Seq,
    GenerationConfig,
    set_seed,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

from trl import SFTTrainer, SFTConfig

from sklearn.metrics import f1_score, classification_report, accuracy_score
from accelerate import Accelerator
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
accelerator = Accelerator(mixed_precision="no")

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("SecurityFT")

torch.backends.cuda.matmul.allow_tf32 = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Device: {DEVICE}")
if DEVICE == "cuda":
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 3 — Master Configuration

In [ ]:

# ─── Dataset Config ───────────────────────────────────────────────────────────
# size_used: number of samples to draw from the full dataset for finetuning.
#            Set to None to use the entire dataset.
#            80% → train, 20% → test split applied after sampling.
#
# task_group values used in this notebook:
#   "qa"             – generative QA (BLEU, ROUGE, Token-F1)
#   "summarization"  – log/report generation (BLEU, ROUGE, Token-F1)
#   "classification" – label prediction (Macro-F1 + Accuracy only)
#
# Unseen benchmarks (purple_team, attackqa) are kept enabled so their
# test splits are evaluated after merged fine-tuning, but they are
# excluded from the merged training corpus via "train": False.

DATASET_CONFIG = {
    # ── Instruction QA ────────────────────────────────────────────────────
    "trendyol": {
        "hf_path": "Trendyol/Trendyol-Cybersecurity-Instruction-Tuning-Dataset",
        "hf_split": "train",
        "size_used": 2000,
        "task_group": "qa",
        "train": True,   # included in merged training corpus
        "enabled": True,
    },
    "security_qna": {
        "hf_path": "Mr-Vicky-01/Security-QnA",
        "hf_split": "train",
        "size_used": 2000,
        "task_group": "qa",
        "train": True,
        "enabled": True,
    },
    "purple_team": {
        # Unseen benchmark — NOT included in training, eval only
        "hf_path": "Canstralian/Purple-Team-Cybersecurity-Dataset",
        "hf_split": "train",
        "size_used": 500,
        "task_group": "qa",
        "train": False,  # excluded from merged training corpus
        "enabled": True,
    },
    # ── Incident Analysis / Log Reasoning ─────────────────────────────────
    "soc_audit": {
        "hf_path": "harleygilpin/soc-audit-11k",
        "hf_split": "train",
        "size_used": 2000,
        "task_group": "summarization",
        "train": True,
        "enabled": True,
    },
    "syslog_artifact": {
        "hf_path": "witfoo/syslog-to-artifact",
        "hf_split": "train",
        "size_used": 2000,
        "task_group": "summarization",
        "train": True,
        "enabled": True,
    },
    # ── Safety Classification ──────────────────────────────────────────────
    "multilingual_jailbreak": {
        "hf_path": "darkknight25/Multilingual_Jailbreak_Dataset",
        "hf_split": "train",
        "size_used": 2000,
        "task_group": "classification",
        "train": True,
        "enabled": True,
    },
    # ── Threat Intelligence / Vulnerability Reasoning ─────────────────────
    "cve_llm": {
        "hf_path": "morpheuslord/cve-llm-training",
        "hf_split": "train",
        "size_used": 2000,
        "task_group": "qa",
        "train": True,
        "enabled": True,
    },
    "mitre_stix": {
        "hf_path": "jason-oneal/mitre-stix-cve-exploitdb-dataset",
        "hf_split": "train",
        "size_used": 2000,
        "task_group": "qa",
        "train": True,
        "enabled": True,
    },
    "attackqa": {
        # Unseen benchmark — NOT included in training, eval only
        "hf_path": "sambanovasystems/AttackQA",
        "hf_split": "train",
        "size_used": 500,
        "task_group": "qa",
        "train": False,  # excluded from merged training corpus
        "enabled": True,
    },
}

# ─── Model Config ─────────────────────────────────────────────────────────────
MODEL_CONFIG = {
    "gemma3": {
        "model_id": "google/gemma-3-270m-it",
        "enabled": True,
        "chat_template": "gemma",
        "max_seq_length": 512,
    },
    "smollm2": {
        "model_id": "HuggingFaceTB/SmolLM2-360M-Instruct",
        "enabled": False,
        "chat_template": "chatml",
        "max_seq_length": 512,
    },
    "qwen3": {
        "model_id": "Qwen/Qwen3-0.6B",
        "enabled": False,
        "chat_template": "chatml",
        "enable_thinking": False,
        "max_seq_length": 512,
    },
}

# ─── LoRA Config (identical for all models — equal comparison) ────────────────
LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": TaskType.CAUSAL_LM,
    "target_modules": "all-linear",
}

# ─── Training Config ──────────────────────────────────────────────────────────
TRAINING_CONFIG = {
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "fp16": False,
    "bf16": False,
    "logging_steps": 25,
    "eval_steps": 100,
    "save_steps": 100,
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "report_to": "tensorboard",
    "dataloader_num_workers": 2,
    "optim": "paged_adamw_8bit",
    "neftune_noise_alpha": 5,
    "max_length": 512,
}

# ─── QLoRA quantisation ───────────────────────────────────────────────────────
BNBCONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_use_double_quant=True,
)

# ─── Paths ────────────────────────────────────────────────────────────────────
BASE_OUTPUT_DIR = "/kaggle/working/checkpoints"
RESULTS_DIR     = "/kaggle/working/results"
LOGS_DIR        = "/kaggle/working/logs"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

TRAIN_TEST_SPLIT = 0.8   # 80 / 20

## Cell 4 — Dataset Loaders & Formatters

In [ ]:

# ─── Prompt template ──────────────────────────────────────────────────────────
def make_prompt(instruction: str, context: str, input_text: str, output: str = "") -> str:
    """Wrap a sample in the unified generation format."""
    prompt = (
        f"<instruction>{instruction.strip()}</instruction>\n"
        f"<context>{context.strip()}</context>\n"
        f"<input>{input_text.strip()}</input>\n"
        f"<output>{output.strip()}</output>"
    )
    return prompt


def make_inference_prompt(instruction: str, context: str, input_text: str) -> str:
    """Inference-time prompt — output tag left open for generation."""
    return (
        f"<instruction>{instruction.strip()}</instruction>\n"
        f"<context>{context.strip()}</context>\n"
        f"<input>{input_text.strip()}</input>\n"
        f"<output>"
    )


# ─── Trendyol Cybersecurity Instruction Dataset ───────────────────────────────
# Fields: "instruction", "input", "output"
# Task: cybersecurity instruction-following QA
def format_trendyol(sample: dict) -> Optional[dict]:
    instruction = str(sample.get("instruction", "")).strip()
    input_text  = str(sample.get("input", "")).strip()
    output      = str(sample.get("output", "")).strip()
    if not instruction or not output:
        return None
    ctx = input_text if input_text else "Cybersecurity instruction-following task."
    return {
        "text": make_prompt(
            instruction=instruction,
            context=ctx,
            input_text=instruction,
            output=output,
        ),
        "task": "qa",
        "dataset": "trendyol",
        "reference": output,
    }


# ─── Security-QnA ─────────────────────────────────────────────────────────────
# Fields: "question", "answer"  (or "input"/"output" depending on version)
# Task: vulnerability-oriented cybersecurity QA
def format_security_qna(sample: dict) -> Optional[dict]:
    question = str(sample.get("question", sample.get("input", ""))).strip()
    answer   = str(sample.get("answer",   sample.get("output", ""))).strip()
    if not question or not answer:
        return None
    return {
        "text": make_prompt(
            instruction=(
                "Answer the following cybersecurity question. Explain the vulnerability, "
                "its exploitation mechanism, and any relevant mitigation strategies."
            ),
            context="Vulnerability-oriented cybersecurity QA.",
            input_text=question,
            output=answer,
        ),
        "task": "qa",
        "dataset": "security_qna",
        "reference": answer,
    }


# ─── Purple-Team Cybersecurity Dataset (unseen benchmark) ─────────────────────
# Fields: "instruction", "input", "output"  (Alpaca-style)
# Task: adversarial/defensive cybersecurity QA — evaluation only
def format_purple_team(sample: dict) -> Optional[dict]:
    instruction = str(sample.get("instruction", "")).strip()
    input_text  = str(sample.get("input", "")).strip()
    output      = str(sample.get("output", "")).strip()
    if not instruction or not output:
        return None
    ctx = input_text if input_text else "Purple-team cybersecurity scenario."
    return {
        "text": make_prompt(
            instruction=instruction,
            context=ctx,
            input_text=instruction,
            output=output,
        ),
        "task": "qa",
        "dataset": "purple_team",
        "reference": output,
    }


# ─── SOC Audit 11K ────────────────────────────────────────────────────────────
# Fields: "input" (raw log), "output" (analyst audit summary)
# Task: security audit and incident report generation
def format_soc_audit(sample: dict) -> Optional[dict]:
    log_input = str(sample.get("input", sample.get("log", ""))).strip()
    summary   = str(sample.get("output", sample.get("summary", ""))).strip()
    if not log_input or not summary:
        return None
    return {
        "text": make_prompt(
            instruction=(
                "Analyse the following security log and generate a concise, human-readable "
                "incident audit summary. Identify key events, anomalies, and any indicators "
                "of compromise."
            ),
            context="Security Operations Centre (SOC) incident analysis.",
            input_text=log_input[:1500],   # guard very long log entries
            output=summary,
        ),
        "task": "summarization",
        "dataset": "soc_audit",
        "reference": summary,
    }


# ─── Syslog-to-Artifact ───────────────────────────────────────────────────────
# Fields: "input" (syslog), "output" (forensic artifact / structured summary)
# Task: log interpretation and forensic artifact extraction
def format_syslog_artifact(sample: dict) -> Optional[dict]:
    syslog   = str(sample.get("input", sample.get("syslog", ""))).strip()
    artifact = str(sample.get("output", sample.get("artifact", ""))).strip()
    if not syslog or not artifact:
        return None
    return {
        "text": make_prompt(
            instruction=(
                "Interpret the following syslog data and extract actionable security artifacts. "
                "Identify suspicious processes, network activity, and any indicators of "
                "compromised system behaviour."
            ),
            context="Forensic syslog analysis and artifact extraction.",
            input_text=syslog[:1500],
            output=artifact,
        ),
        "task": "summarization",
        "dataset": "syslog_artifact",
        "reference": artifact,
    }


# ─── Multilingual Jailbreak Dataset ──────────────────────────────────────────
# Fields: "prompt" (adversarial input), "label" ("safe" / "unsafe")
# Task: adversarial prompt safety classification
# NOTE: classification task — only Macro-F1 and Accuracy are reported;
#       BLEU/ROUGE are NOT computed for this dataset.
_JAILBREAK_LABEL_MAP = {"safe": 0, "unsafe": 1, 0: 0, 1: 1}

def format_multilingual_jailbreak(sample: dict) -> Optional[dict]:
    prompt = str(sample.get("prompt", sample.get("text", ""))).strip()
    raw_label = sample.get("label", sample.get("class", None))
    if not prompt or raw_label is None:
        return None
    # Normalise to string label
    if isinstance(raw_label, int):
        label_str = "unsafe" if raw_label == 1 else "safe"
    else:
        label_str = str(raw_label).strip().lower()
    if label_str not in ("safe", "unsafe"):
        return None
    return {
        "text": make_prompt(
            instruction=(
                "Classify the following prompt as either safe or unsafe. "
                "Respond with exactly one word: safe or unsafe."
            ),
            context="Adversarial prompt safety classification.",
            input_text=prompt,
            output=label_str,
        ),
        "task": "classification",
        "dataset": "multilingual_jailbreak",
        "reference": label_str,
    }


# ─── CVE-LLM Training Dataset ─────────────────────────────────────────────────
# Fields: "instruction", "input", "output"
# Task: vulnerability reasoning and mitigation QA
def format_cve_llm(sample: dict) -> Optional[dict]:
    instruction = str(sample.get("instruction", "")).strip()
    input_text  = str(sample.get("input", "")).strip()
    output      = str(sample.get("output", "")).strip()
    if not output:
        return None
    instr = instruction if instruction else (
        "Analyse the following CVE record. Explain the vulnerability, its exploitation "
        "impact, affected systems, and recommended mitigations."
    )
    query = input_text if input_text else instruction
    return {
        "text": make_prompt(
            instruction=instr,
            context="CVE vulnerability analysis and mitigation.",
            input_text=query,
            output=output,
        ),
        "task": "qa",
        "dataset": "cve_llm",
        "reference": output,
    }


# ─── MITRE-STIX-CVE-ExploitDB Dataset ─────────────────────────────────────────
# Fields: "instruction", "input", "output"  (may also have "context")
# Task: threat intelligence reasoning across MITRE ATT&CK, STIX, CVE, ExploitDB
def format_mitre_stix(sample: dict) -> Optional[dict]:
    instruction = str(sample.get("instruction", "")).strip()
    input_text  = str(sample.get("input", "")).strip()
    extra_ctx   = str(sample.get("context", "")).strip()
    output      = str(sample.get("output", "")).strip()
    if not output:
        return None
    instr = instruction if instruction else (
        "Using the provided threat intelligence context, answer the question about the "
        "vulnerability, exploit, or adversarial tactic described."
    )
    ctx = extra_ctx if extra_ctx else "MITRE ATT&CK / STIX / CVE / ExploitDB threat intelligence."
    query = input_text if input_text else instruction
    return {
        "text": make_prompt(
            instruction=instr,
            context=ctx,
            input_text=query,
            output=output,
        ),
        "task": "qa",
        "dataset": "mitre_stix",
        "reference": output,
    }


# ─── AttackQA (unseen benchmark) ──────────────────────────────────────────────
# Fields: "question", "answer"  (or "context" for passage-based QA)
# Task: unseen threat intelligence QA — evaluation only
def format_attackqa(sample: dict) -> Optional[dict]:
    question = str(sample.get("question", sample.get("input", ""))).strip()
    answer   = str(sample.get("answer",   sample.get("output", ""))).strip()
    passage  = str(sample.get("context",  "")).strip()
    if not question or not answer:
        return None
    ctx = passage[:800] if passage else "ATT&CK-style adversarial threat intelligence."
    return {
        "text": make_prompt(
            instruction=(
                "Answer the following threat intelligence question based on your knowledge "
                "of adversarial tactics, techniques, and procedures."
            ),
            context=ctx,
            input_text=question,
            output=answer,
        ),
        "task": "qa",
        "dataset": "attackqa",
        "reference": answer,
    }

## Cell 5 — Dataset Loading Pipeline

In [ ]:

def _sample_dataset(ds: Dataset, size_used: Optional[int], seed: int = 42) -> Dataset:
    """Draw `size_used` samples; if None, return full dataset."""
    if size_used is None or size_used >= len(ds):
        return ds
    return ds.shuffle(seed=seed).select(range(size_used))


def _apply_formatter_flat(ds: Dataset, formatter) -> List[dict]:
    """Apply a formatter that returns Optional[dict] (one sample → one output)."""
    results = []
    for sample in tqdm(ds, desc="Formatting", leave=False):
        out = formatter(sample)
        if out is not None:
            results.append(out)
    return results


def load_and_format_all(dataset_config: dict) -> Dict[str, List[dict]]:
    """
    Load every enabled dataset, apply size limit, format to unified schema.
    Returns dict[dataset_key → list of formatted samples].
    """
    formatted = {}

    for key, cfg in dataset_config.items():
        if not cfg.get("enabled", True):
            logger.info(f"Skipping {key} (disabled)")
            continue

        logger.info(f"Loading dataset: {key}")

        try:
            raw = load_dataset(cfg["hf_path"], split=cfg.get("hf_split", "train"))
            logger.info(f"  Raw size: {len(raw)}")

            raw = _sample_dataset(raw, cfg.get("size_used"))
            logger.info(f"  After sampling: {len(raw)}")

            # ── Format ──────────────────────────────────────────────────────
            formatter_map = {
                "trendyol":             format_trendyol,
                "security_qna":         format_security_qna,
                "purple_team":          format_purple_team,
                "soc_audit":            format_soc_audit,
                "syslog_artifact":      format_syslog_artifact,
                "multilingual_jailbreak": format_multilingual_jailbreak,
                "cve_llm":              format_cve_llm,
                "mitre_stix":           format_mitre_stix,
                "attackqa":             format_attackqa,
            }

            if key not in formatter_map:
                logger.warning(f"  Unknown key {key}. Skipping.")
                continue

            samples = _apply_formatter_flat(raw, formatter_map[key])
            logger.info(f"  Formatted samples: {len(samples)}")
            formatted[key] = samples

        except Exception as e:
            logger.error(f"  Failed to load/format {key}: {e}")
            continue

    return formatted


def split_dataset(samples: List[dict], train_ratio: float = TRAIN_TEST_SPLIT, seed: int = 42
                  ) -> Tuple[List[dict], List[dict]]:
    """80/20 deterministic split."""
    random.seed(seed)
    shuffled = samples.copy()
    random.shuffle(shuffled)
    cut = int(len(shuffled) * train_ratio)
    return shuffled[:cut], shuffled[cut:]


def build_dataset_splits(formatted: Dict[str, List[dict]]) -> Dict[str, Dict[str, List[dict]]]:
    """Returns {dataset_key: {'train': [...], 'test': [...]}}"""
    splits = {}
    for key, samples in formatted.items():
        train, test = split_dataset(samples)
        splits[key] = {"train": train, "test": test}
        logger.info(f"  {key}: {len(train)} train | {len(test)} test")
    return splits


# ─── Run loaders ──────────────────────────────────────────────────────────────
logger.info("=" * 60)
logger.info("Loading and formatting all datasets")
logger.info("=" * 60)
ALL_FORMATTED  = load_and_format_all(DATASET_CONFIG)
DATASET_SPLITS = build_dataset_splits(ALL_FORMATTED)

## Cell 6 — Model Factory & LoRA Setup

In [ ]:

def get_target_modules(model_key: str) -> List[str]:
    """
    Return LoRA target module names per architecture.
    All models → attention projections + gate/up/down proj for equal footing.
    """
    if model_key == "smollm2":
        return ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]
    elif model_key == "qwen3":
        return ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]
    elif model_key == "gemma3":
        return ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]
    return ["q_proj", "v_proj"]


def load_model_and_tokenizer(model_key: str, model_cfg: dict):
    """Load quantised model + tokenizer, attach LoRA adapter."""
    model_id = model_cfg["model_id"]
    logger.info(f"Loading model: {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        padding_side="right",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=BNBCONFIG,
        device_map={"": 0},
        dtype=torch.float32,
        attn_implementation="eager"
    )

    model = prepare_model_for_kbit_training(model)

    lora_cfg = LoraConfig(
        r=LORA_CONFIG["r"],
        lora_alpha=LORA_CONFIG["lora_alpha"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        bias=LORA_CONFIG["bias"],
        task_type=LORA_CONFIG["task_type"],
        target_modules=get_target_modules(model_key),
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    return model, tokenizer


def apply_chat_template(model_key: str, tokenizer, sample_text: str) -> str:
    """
    Wrap the unified prompt in the model-specific chat format
    so the model sees the correct special tokens.
    """
    if "<output>" in sample_text and "</output>" in sample_text:
        prefix = sample_text[:sample_text.rfind("<output>") + len("<output>")]
        output = sample_text.split("<output>")[-1].replace("</output>", "").strip()
    else:
        prefix = sample_text
        output = ""

    if model_key in ("smollm2", "qwen3"):
        full = (
            f"<|im_start|>user\n{prefix}<|im_end|>\n"
            f"<|im_start|>assistant\n{output}<|im_end|>"
        )
    elif model_key == "gemma3":
        full = (
            f"<start_of_turn>user\n{prefix}<end_of_turn>\n"
            f"<start_of_turn>model\n{output}<end_of_turn>"
        )
    else:
        full = sample_text

    return full


def tokenize_dataset(model_key: str, tokenizer, samples: List[dict],
                      max_length: int) -> Dataset:
    """Convert list of formatted samples to a tokenised HF Dataset."""
    texts = [apply_chat_template(model_key, tokenizer, s["text"]) for s in samples]
    ds    = Dataset.from_dict({"text": texts})
    return ds

## Cell 7 — Checkpoint & Logging Callbacks

In [ ]:

class SecurityTrainingCallback(TrainerCallback):
    """
    Custom callback:
    - Logs train/eval loss to a JSONL file per run.
    - Prints a summary line every `log_every` steps.
    - Saves best checkpoint path to a text file.
    """

    def __init__(self, log_path: str, log_every: int = 25):
        self.log_path  = log_path
        self.log_every = log_every
        self.best_eval = float("inf")
        self._fh = open(log_path, "w")

    def _write(self, record: dict):
        self._fh.write(json.dumps(record) + "\n")
        self._fh.flush()

    def on_log(self, args, state: TrainerState, control: TrainerControl, logs=None, **kwargs):
        if logs is None:
            return
        record = {"step": state.global_step, "epoch": state.epoch, **logs}
        self._write(record)
        if state.global_step % self.log_every == 0:
            parts = [f"step={state.global_step}", f"epoch={state.epoch:.2f}"]
            for k in ("loss", "eval_loss", "learning_rate"):
                if k in logs:
                    parts.append(f"{k}={logs[k]:.4f}")
            logger.info("  [Train] " + " | ".join(parts))

    def on_evaluate(self, args, state: TrainerState, control: TrainerControl, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            el = metrics["eval_loss"]
            if el < self.best_eval:
                self.best_eval = el
                logger.info(f"  ✓ New best eval_loss={el:.4f} at step {state.global_step}")

    def on_train_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        self._fh.close()
        logger.info(f"  Training log saved to {self.log_path}")

## Cell 8 — Training Function (Generic)

In [ ]:

def run_training(
    model_key: str,
    run_name: str,
    train_samples: List[dict],
    eval_samples: List[dict],
    output_dir: str,
    train: bool,
):
    """
    Full training run for a given model + dataset split.
    Handles model loading, tokenisation, SFTTrainer, checkpointing.
    Returns path to best checkpoint.
    """
    model_cfg = MODEL_CONFIG[model_key]
    if not model_cfg["enabled"]:
        logger.info(f"Model {model_key} disabled. Skipping.")
        return None

    os.makedirs(output_dir, exist_ok=True)
    log_path = os.path.join(LOGS_DIR, f"{run_name}.jsonl")

    logger.info("=" * 60)
    logger.info(f"RUN: {run_name}")
    logger.info(f"  Model   : {model_cfg['model_id']}")
    logger.info(f"  Train   : {len(train_samples)} | Eval: {len(eval_samples)}")
    logger.info("=" * 60)

    model, tokenizer = load_model_and_tokenizer(model_key, model_cfg)

    train_ds = tokenize_dataset(model_key, tokenizer, train_samples,
                                model_cfg["max_seq_length"])
    eval_ds  = tokenize_dataset(model_key, tokenizer, eval_samples,
                                model_cfg["max_seq_length"])

    training_args = SFTConfig(
        output_dir=output_dir,
        run_name=run_name,
        num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
        per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
        per_device_eval_batch_size=TRAINING_CONFIG["per_device_eval_batch_size"],
        gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
        learning_rate=TRAINING_CONFIG["learning_rate"],
        lr_scheduler_type=TRAINING_CONFIG["lr_scheduler_type"],
        warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
        weight_decay=TRAINING_CONFIG["weight_decay"],
        fp16=TRAINING_CONFIG["fp16"],
        bf16=TRAINING_CONFIG["bf16"],
        logging_steps=TRAINING_CONFIG["logging_steps"],
        eval_steps=TRAINING_CONFIG["eval_steps"],
        save_steps=TRAINING_CONFIG["save_steps"],
        save_total_limit=TRAINING_CONFIG["save_total_limit"],
        load_best_model_at_end=TRAINING_CONFIG["load_best_model_at_end"],
        metric_for_best_model=TRAINING_CONFIG["metric_for_best_model"],
        greater_is_better=TRAINING_CONFIG["greater_is_better"],
        report_to=TRAINING_CONFIG["report_to"],
        logging_dir=os.path.join(LOGS_DIR, "tensorboard", run_name),
        dataloader_num_workers=TRAINING_CONFIG["dataloader_num_workers"],
        optim=TRAINING_CONFIG["optim"],
        neftune_noise_alpha=TRAINING_CONFIG["neftune_noise_alpha"],
        eval_strategy="steps",
        dataset_text_field="text",
        packing=False,
        max_length=TRAINING_CONFIG["max_length"],
    )

    callback = SecurityTrainingCallback(log_path=log_path)

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        args=training_args,
        callbacks=[callback],
    )

    logger.info("Starting training...")
    if train:
        trainer.train()
    logger.info("Finished training...")

    best_ckpt = trainer.state.best_model_checkpoint
    logger.info(f"Best checkpoint: {best_ckpt}")

    adapter_path = os.path.join(output_dir, "final_adapter")
    trainer.model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    logger.info(f"Adapter saved to: {adapter_path}")

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return adapter_path

## Cell 9 — Evaluation Utilities

In [ ]:

bleu_metric  = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")


def _strip_output_tag(text: str) -> str:
    """Extract only the text inside <output>...</output>."""
    m = re.search(r"<output>(.*?)(?:</output>|$)", text, re.DOTALL)
    return m.group(1).strip() if m else text.strip()


def generate_predictions(
    model_key: str,
    adapter_path: str,
    test_samples: List[dict],
    batch_size: int = 8,
    max_new_tokens: int = 150,
) -> Tuple[List[str], List[str]]:
    """
    Load a trained LoRA adapter and generate predictions on test_samples.
    Returns (predictions, references).
    """
    model_cfg = MODEL_CONFIG[model_key]
    logger.info(f"Generating predictions: {adapter_path}")

    tokenizer = AutoTokenizer.from_pretrained(adapter_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    base_model = AutoModelForCausalLM.from_pretrained(
        model_cfg["model_id"],
        quantization_config=BNBCONFIG,
        device_map={"": 0},
        dtype=torch.float32,
        attn_implementation="eager"
    )
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()

    predictions, references = [], []

    for i in tqdm(range(0, len(test_samples), batch_size), desc="Generating"):
        batch = test_samples[i : i + batch_size]

        prompts = []
        for s in batch:
            raw = s["text"]
            if "<output>" in raw:
                raw = raw[:raw.rfind("<output>") + len("<output>")]
            if model_key in ("smollm2", "qwen3"):
                p = f"<|im_start|>user\n{raw}<|im_end|>\n<|im_start|>assistant\n"
            elif model_key == "gemma3":
                p = f"<start_of_turn>user\n{raw}<end_of_turn>\n<start_of_turn>model\n"
            else:
                p = raw
            prompts.append(p)

        enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MODEL_CONFIG[model_key]["max_seq_length"],
        ).to(DEVICE)

        with torch.no_grad():
            out_ids = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.3,
                top_p=0.4,
                repetition_penalty=1.2,
            )

        input_len = enc["input_ids"].shape[1]
        for j, ids in enumerate(out_ids):
            pred_text = tokenizer.decode(ids[input_len:], skip_special_tokens=True).strip()
            predictions.append(pred_text)
            references.append(str(batch[j].get("reference", "")))

    del model, base_model
    gc.collect()
    torch.cuda.empty_cache()

    return predictions, references


def compute_bleu(predictions: List[str], references: List[str]) -> float:
    refs_wrapped = [[r] for r in references]
    result = bleu_metric.compute(predictions=predictions, references=refs_wrapped)
    return round(result["score"], 4)


def compute_rouge(predictions: List[str], references: List[str]) -> dict:
    result = rouge_metric.compute(predictions=predictions, references=references)
    return {k: round(v, 4) for k, v in result.items()}


def compute_token_f1(prediction: str, reference: str) -> float:
    """Token-level F1 (standard QA metric)."""
    pred_tokens = prediction.lower().split()
    ref_tokens  = reference.lower().split()
    common = set(pred_tokens) & set(ref_tokens)
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens) if pred_tokens else 0.0
    recall    = len(common) / len(ref_tokens)  if ref_tokens  else 0.0
    if precision + recall == 0:
        return 0.0
    return round(2 * precision * recall / (precision + recall), 4)


def compute_classification_metrics(predictions: List[str], references: List[str]) -> dict:
    """
    Macro-F1 + Accuracy for binary safety classification (safe / unsafe).
    NOTE: BLEU and ROUGE are intentionally NOT computed for classification tasks.
    """
    label_map = {"safe": 0, "unsafe": 1}
    y_true, y_pred = [], []
    for p, r in zip(predictions, references):
        # Take the first word of the prediction, strip punctuation
        p_clean = p.lower().strip().split()[0] if p.strip() else "safe"
        p_clean = p_clean.strip(".,;:")
        y_true.append(label_map.get(r.lower(), 0))
        y_pred.append(label_map.get(p_clean, 0))
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    acc      = accuracy_score(y_true, y_pred)
    report   = classification_report(
        y_true, y_pred,
        target_names=["safe", "unsafe"],
        output_dict=True,
        zero_division=0,
    )
    return {
        "macro_f1":  round(macro_f1, 4),
        "accuracy":  round(acc, 4),
        "safe_f1":   round(report["safe"]["f1-score"],   4),
        "unsafe_f1": round(report["unsafe"]["f1-score"], 4),
    }


def evaluate_run(
    model_key: str,
    adapter_path: str,
    test_samples: List[dict],
    task_group: str,
    run_label: str,
) -> dict:
    """
    Generate and evaluate predictions for a run.
    Metric selection is gated by task_group:
      - qa / summarization  → BLEU, ROUGE-1/2/L, Token-F1
      - classification      → Macro-F1, Accuracy only
    """
    predictions, references = generate_predictions(model_key, adapter_path, test_samples)

    results = {
        "run":        run_label,
        "model":      model_key,
        "task_group": task_group,
        "n_samples":  len(test_samples),
    }

    if task_group in ("qa", "summarization"):
        # Generation metrics
        results["bleu"] = compute_bleu(predictions, references)
        rouge = compute_rouge(predictions, references)
        results.update(rouge)
        token_f1s = [compute_token_f1(p, r) for p, r in zip(predictions, references)]
        results["token_f1_mean"] = round(float(np.mean(token_f1s)), 4)

    elif task_group == "classification":
        # Classification metrics only — no BLEU/ROUGE
        clf_metrics = compute_classification_metrics(predictions, references)
        results.update(clf_metrics)

    # Save raw predictions
    pred_path = os.path.join(RESULTS_DIR, f"{run_label}_predictions.jsonl")
    with open(pred_path, "w") as f:
        for p, r in zip(predictions, references):
            f.write(json.dumps({"prediction": p, "reference": r}) + "\n")

    logger.info(f"  Results for {run_label}: {results}")
    return results

## Cell 10 — Benchmark A: Merged Finetuning

In [ ]:

def run_merged_benchmark(train=True):
    """
    Benchmark A: Merge all training-eligible dataset splits, finetune each
    model once, evaluate each dataset's test split independently.

    Unseen benchmarks (purple_team, attackqa) are excluded from the merged
    training corpus (train=False in DATASET_CONFIG) but are still evaluated
    after fine-tuning to test out-of-distribution generalisation.
    """
    logger.info("\n" + "=" * 60)
    logger.info("BENCHMARK A — MERGED FINETUNING")
    logger.info("=" * 60)

    # Build merged train set — only include datasets marked train=True
    all_train, all_eval = [], []
    for key, split in DATASET_SPLITS.items():
        if DATASET_CONFIG[key].get("train", True):
            all_train.extend(split["train"])
            all_eval.extend(split["test"][:100])

    random.shuffle(all_train)
    random.shuffle(all_eval)
    logger.info(f"Merged train: {len(all_train)} | Merged eval: {len(all_eval)}")

    merged_results = []

    for model_key, model_cfg in MODEL_CONFIG.items():
        if not model_cfg["enabled"]:
            continue

        run_name   = f"merged_{model_key}"
        output_dir = os.path.join(BASE_OUTPUT_DIR, run_name)

        adapter_path = run_training(
            model_key=model_key,
            run_name=run_name,
            train_samples=all_train,
            eval_samples=all_eval,
            output_dir=output_dir,
            train=train,
        )
        if adapter_path is None:
            continue

        # Evaluate per-dataset on its own test split (seen + unseen)
        for ds_key, split in DATASET_SPLITS.items():
            task_group = DATASET_CONFIG[ds_key]["task_group"]
            run_label  = f"merged_{model_key}_on_{ds_key}"
            res = evaluate_run(
                model_key=model_key,
                adapter_path=adapter_path,
                test_samples=split["test"],
                task_group=task_group,
                run_label=run_label,
            )
            res["benchmark"] = "merged"
            res["unseen"]    = not DATASET_CONFIG[ds_key].get("train", True)
            merged_results.append(res)

    out_path = os.path.join(RESULTS_DIR, "benchmark_A_merged.json")
    with open(out_path, "w") as f:
        json.dump(merged_results, f, indent=2)
    logger.info(f"Benchmark A results saved: {out_path}")
    return merged_results

## Cell 11 — Benchmark B: Individual Dataset Finetuning

In [ ]:

def run_individual_benchmark(train=True):
    """
    Benchmark B: For each (model, dataset) pair, finetune from scratch
    and evaluate on that dataset's own test split.
    Unseen benchmarks are skipped in training but still evaluated.
    """
    logger.info("\n" + "=" * 60)
    logger.info("BENCHMARK B — INDIVIDUAL DATASET FINETUNING")
    logger.info("=" * 60)

    individual_results = []

    for ds_key, split in DATASET_SPLITS.items():
        task_group    = DATASET_CONFIG[ds_key]["task_group"]
        train_samples = split["train"]
        test_samples  = split["test"]
        eval_slice    = test_samples[:min(100, len(test_samples))]

        for model_key, model_cfg in MODEL_CONFIG.items():
            if not model_cfg["enabled"]:
                continue

            run_name   = f"indiv_{model_key}_{ds_key}"
            output_dir = os.path.join(BASE_OUTPUT_DIR, run_name)

            adapter_path = run_training(
                model_key=model_key,
                run_name=run_name,
                train_samples=train_samples,
                eval_samples=eval_slice,
                output_dir=output_dir,
                train=train,
            )
            if adapter_path is None:
                continue

            run_label = f"indiv_{model_key}_{ds_key}"
            res = evaluate_run(
                model_key=model_key,
                adapter_path=adapter_path,
                test_samples=test_samples,
                task_group=task_group,
                run_label=run_label,
            )
            res["benchmark"] = "individual"
            res["dataset"]   = ds_key
            individual_results.append(res)

    out_path = os.path.join(RESULTS_DIR, "benchmark_B_individual.json")
    with open(out_path, "w") as f:
        json.dump(individual_results, f, indent=2)
    logger.info(f"Benchmark B results saved: {out_path}")
    return individual_results

## Cell 12 — Results Aggregation & Summary Tables

In [ ]:

def build_summary_table(results: List[dict], label: str) -> pd.DataFrame:
    """Convert a list of result dicts to a readable DataFrame."""
    rows = []
    for r in results:
        row = {
            "Benchmark":  r.get("benchmark", label),
            "Model":      r.get("model", ""),
            "Dataset":    r.get("run", "").split("_on_")[-1] if "on_" in r.get("run", "") else r.get("dataset", ""),
            "Task":       r.get("task_group", ""),
            "Unseen":     r.get("unseen", False),
            "N":          r.get("n_samples", ""),
            # Generation metrics (qa / summarization)
            "BLEU":       r.get("bleu", ""),
            "ROUGE-1":    r.get("rouge1", ""),
            "ROUGE-2":    r.get("rouge2", ""),
            "ROUGE-L":    r.get("rougeL", ""),
            "Token-F1":   r.get("token_f1_mean", ""),
            # Classification metrics (classification only)
            "Macro-F1":   r.get("macro_f1", ""),
            "Accuracy":   r.get("accuracy", ""),
        }
        rows.append(row)
    df = pd.DataFrame(rows)
    return df


def print_summary(
    merged_results_without_training: List[dict],
    merged_results: List[dict],
    individual_results: List[dict],
):
    logger.info("\n" + "=" * 60)
    logger.info("FINAL RESULTS SUMMARY")
    logger.info("=" * 60)

    df_merged          = build_summary_table(merged_results, "merged")
    df_merged_no_train = build_summary_table(merged_results_without_training, "merged_no_train")
    df_all             = pd.concat([df_merged, df_merged_no_train], ignore_index=True)

    csv_path = os.path.join(RESULTS_DIR, "all_results.csv")
    df_all.to_csv(csv_path, index=False)
    logger.info(f"Full results table: {csv_path}")

    # Per-model summaries
    for model_key in MODEL_CONFIG:
        sub = df_all[df_all["Model"] == model_key]
        if sub.empty:
            continue
        logger.info(f"\n── {model_key.upper()} ──")
        print(sub[[
            "Benchmark", "Dataset", "Task", "Unseen",
            "BLEU", "ROUGE-1", "ROUGE-L",
            "Token-F1", "Macro-F1", "Accuracy"
        ]].to_string(index=False))

    # Per-task-group averages
    logger.info("\n── Per-Task Averages (numeric columns) ──")
    numeric_cols = ["BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "Token-F1", "Macro-F1", "Accuracy"]
    for col in numeric_cols:
        df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

    agg = (
        df_all.groupby(["Benchmark", "Model", "Task"])[numeric_cols]
        .mean()
        .round(4)
        .reset_index()
    )
    print(agg.to_string(index=False))

    agg_path = os.path.join(RESULTS_DIR, "aggregated_results.csv")
    agg.to_csv(agg_path, index=False)
    logger.info(f"Aggregated results: {agg_path}")

    return df_all, agg

## Cell 13 — Main Entry Point

In [ ]:

def main():
    logger.info("Security LLM Finetuning Pipeline starting...")
    logger.info(f"Timestamp: {datetime.now().isoformat()}")
    logger.info(f"Enabled datasets : {[k for k, v in DATASET_CONFIG.items() if v.get('enabled')]}")
    logger.info(f"Training datasets: {[k for k, v in DATASET_CONFIG.items() if v.get('train') and v.get('enabled')]}")
    logger.info(f"Unseen benchmarks: {[k for k, v in DATASET_CONFIG.items() if not v.get('train') and v.get('enabled')]}")
    logger.info(f"Enabled models   : {[k for k, v in MODEL_CONFIG.items() if v.get('enabled')]}")

    # ── Benchmark A: Merged ──────────────────────────────────────────────────
    # Zero-shot baseline (no training)
    merged_results_without_training = run_merged_benchmark(train=False)
    # Fine-tuned
    merged_results_with_training = run_merged_benchmark(train=True)

    # ── Benchmark B: Individual ──────────────────────────────────────────────
    # individual_results = run_individual_benchmark()

    # ── Summary ──────────────────────────────────────────────────────────────
    df_all, agg = print_summary(
        merged_results_without_training,
        merged_results_with_training,
        [],
    )

    logger.info("\nAll done. Outputs written to /kaggle/working/results/")
    return df_all, agg


if __name__ == "__main__":
    try:
        df_all, agg = main()
    except Exception as e:
        logger.error(f"An error occurred during pipeline execution: {e}")
        import traceback
        traceback.print_exc()